# NB6057CEM — Artificial Neural Networks
## Portfolio Upgrade: Phase 3 — Temporal Deep Learning (CNN + GRU Hybrid Audio Classifier)
**Compute Target:** Google Colab Pro with CUDA GPU Acceleration (NVIDIA T4 / V100 / A100)

This self-contained notebook trains and evaluates the **CNN + GRU Hybrid (CRNN)** architecture on the official UrbanSound8K dataset:
- **Temporal Log-Mel Spectrogram Representation:** `(1, 128, 173)` retaining 173 time frames across 128 mel frequency channels.
- **Hybrid Architecture:** 3-stage 2D CNN feature extractor $\to$ Adaptive frequency collapse $\to$ Bidirectional GRU ($2 \times 64 = 128$) $\to$ Temporal mean pooling $\to$ Binary siren classifier (77,601 trainable parameters).
- **Leakage-Free 10-Fold Protocol:** Folds 1–8 for training ($N=7,095$), Fold 9 for validation ($N=800$), and Fold 10 for held-out testing ($N=837$).
- **Google Drive Persistence:** Checkpoints (`cnn_gru_seed42.pt`), learning curves, and benchmark metrics are persisted directly to Google Drive (`/content/drive/MyDrive/ANN-Siren-Detection/`).

In [ ]:
# CELL 1: Environment & GPU Hardware Inspection
import os
import platform
import sys
from pathlib import Path
import torch

print('=' * 75)
print('SYSTEM ENVIRONMENT & HARDWARE ACCELERATOR INSPECTION')
print('=' * 75)
print(f'Platform            : {platform.system()} {platform.release()} ({platform.machine()})')
print(f'Python Executable   : {sys.executable}')
print(f'Python Version      : {platform.python_version()}')
print(f'PyTorch Version     : {torch.__version__}')
print(f'CUDA Available      : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    device = torch.device('cuda:0')
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    capability = torch.cuda.get_device_capability(0)
    print(f'GPU Device Name     : {gpu_name}')
    print(f'Allocated VRAM      : {vram_gb:.2f} GB')
    print(f'Compute Capability  : {capability[0]}.{capability[1]}')
    print(f'cuDNN Enabled       : {torch.backends.cudnn.enabled}')
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device('cpu')
    print('[WARN] CUDA is not available. Please navigate to:')
    print('       Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU / V100 GPU / A100 GPU')

# Mount Google Drive
GDRIVE_DIR = None
try:
    from google.colab import drive
    mount_path = '/content/drive'
    if not Path(mount_path, 'MyDrive').is_dir():
        drive.mount(mount_path)
    GDRIVE_DIR = Path(mount_path) / 'MyDrive' / 'ANN-Siren-Detection'
    GDRIVE_DIR.mkdir(parents=True, exist_ok=True)
    (GDRIVE_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
    (GDRIVE_DIR / 'results' / 'metrics').mkdir(parents=True, exist_ok=True)
    (GDRIVE_DIR / 'results' / 'figures').mkdir(parents=True, exist_ok=True)
    print(f'Google Drive Root   : {GDRIVE_DIR}')
except Exception as e:
    print(f'[INFO] Google Drive mounting skipped or running locally: {e}')
print('=' * 75)

In [ ]:
# CELL 2: Project Workspace Bootstrap & Source File Extraction
import base64
import hashlib
import io
import zipfile
from pathlib import Path

WORKSPACE = Path('/content/ANN-Siren-Detection') if Path('/content').is_dir() else Path.cwd()
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

print(f'Active Workspace: {WORKSPACE}')

# Check if Drive package zip exists
drive_pkg = GDRIVE_DIR / 'colab_phase3_package.zip' if GDRIVE_DIR else None
if drive_pkg and drive_pkg.is_file():
    print(f'Unpacking transfer package from Google Drive: {drive_pkg}...')
    with zipfile.ZipFile(drive_pkg, 'r') as z:
        z.extractall(WORKSPACE)
    print('Transfer package successfully unpacked.')
else:
    # Unpack embedded project code bundle
    print('Extracting embedded project code bundle...')
    BUNDLE_B64 = "..."
    raw_bytes = base64.b64decode(BUNDLE_B64)
    with zipfile.ZipFile(io.BytesIO(raw_bytes), 'r') as z:
        z.extractall(WORKSPACE)
    print('Embedded source modules extracted successfully.')

print('\nWorkspace directory structure:')
for d in ('configs', 'src', 'scripts'):
    p = WORKSPACE / d
    if p.is_dir():
        files = [f.name for f in p.iterdir() if f.is_file()]
        print(f'  {d}/ ({len(files)} files): {files[:6]}...')

In [ ]:
# CELL 3: Install and Verify Python Dependencies
!pip install -q soundfile librosa pyyaml tabulate scikit-learn

import yaml
import librosa
import soundfile
import sklearn
import matplotlib

print('Dependency verification:')
print(f'  PyTorch    : {torch.__version__}')
import torchaudio
print(f'  Torchaudio : {torchaudio.__version__}')
print(f'  Librosa    : {librosa.__version__}')
print(f'  SoundFile  : {soundfile.__version__}')
print(f'  Scikit-Learn: {sklearn.__version__}')
print(f'  PyYAML     : {yaml.__version__}')
print('All required libraries imported cleanly.')

In [ ]:
# CELL 4: Data Validation & Split Leakage Protection
import shutil
import numpy as np
import zipfile

processed_dir = WORKSPACE / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

train_val_cache = processed_dir / 'train_val_mel_cache.npz'
test_cache = processed_dir / 'test_mel_cache.npz'

# If cache is on Google Drive, copy or link to local fast disk
if GDRIVE_DIR and (not train_val_cache.is_file() or not test_cache.is_file()):
    drive_tv = GDRIVE_DIR / 'data' / 'processed' / 'train_val_mel_cache.npz'
    drive_test = GDRIVE_DIR / 'data' / 'processed' / 'test_mel_cache.npz'
    if drive_tv.is_file() and not train_val_cache.is_file():
        print(f'Copying {drive_tv.name} from Google Drive to local fast scratch...')
        shutil.copy2(drive_tv, train_val_cache)
    if drive_test.is_file() and not test_cache.is_file():
        print(f'Copying {drive_test.name} from Google Drive to local fast scratch...')
        shutil.copy2(drive_test, test_cache)

assert train_val_cache.is_file(), f'Training cache missing: {train_val_cache}'
assert test_cache.is_file(), f'Test cache missing: {test_cache}'

print('=' * 75)
print('DATASET VALIDATION & TENSOR SHAPES')
print('=' * 75)

with zipfile.ZipFile(train_val_cache) as z:
    with z.open('y_train.npy') as f:
        y_train = np.load(f)
    with z.open('y_val.npy') as f:
        y_val = np.load(f)

with zipfile.ZipFile(test_cache) as z:
    with z.open('y_test.npy') as f:
        y_test = np.load(f)

# Extract OS-level memory-mapped buffers for zero-copy training
from src.train_hybrid import ensure_mmap_mel_files
train_mmap, val_mmap, test_mmap = ensure_mmap_mel_files(processed_dir)
x_train_arr = np.load(train_mmap, mmap_mode='r')
x_val_arr = np.load(val_mmap, mmap_mode='r')
x_test_arr = np.load(test_mmap, mmap_mode='r')

print(f'Training Set (Folds 1-8) : X={x_train_arr.shape}, y={y_train.shape} | Sirens={np.sum(y_train==1)} ({np.mean(y_train==1)*100:.2f}%)')
print(f'Validation Set (Fold 9)  : X={x_val_arr.shape}, y={y_val.shape}   | Sirens={np.sum(y_val==1)} ({np.mean(y_val==1)*100:.2f}%)')
print(f'Held-Out Test (Fold 10)  : X={x_test_arr.shape}, y={y_test.shape}   | Sirens={np.sum(y_test==1)} ({np.mean(y_test==1)*100:.2f}%)')

# Strict verification assertions
assert x_train_arr.shape == (7095, 1, 128, 173), f'Unexpected train shape: {x_train_arr.shape}'
assert x_val_arr.shape == (800, 1, 128, 173), f'Unexpected val shape: {x_val_arr.shape}'
assert x_test_arr.shape == (837, 1, 128, 173), f'Unexpected test shape: {x_test_arr.shape}'
print('\nSplit Leakage Verification: Fold 10 test data is strictly isolated.')
print('All tensor dimensions and sample counts conform exactly to UrbanSound8K protocol.')
print('=' * 75)

In [ ]:
# CELL 5: Phase 3 CNN + GRU Hybrid Training (CUDA Accelerated)
import time
from src.hybrid_model import CNNGRUHybrid
from src.train_hybrid import train_cnn_gru_model

# Training Configuration
SEEDS = [42]             # Random seeds to train (can expand to [42, 123, 2026])
BATCH_SIZE = 64          # Mini-batch size for GPU execution
MAX_EPOCHS = 25          # Maximum training epochs
PATIENCE = 6             # Early stopping patience tracking Fold 9 Macro-F1
LEARNING_RATE = 1e-3     # Adam optimizer learning rate
WEIGHT_DECAY = 1e-4      # L2 regularization

compute_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoints_dir = WORKSPACE / 'checkpoints'
figures_dir = WORKSPACE / 'results' / 'figures'
metrics_dir = WORKSPACE / 'results' / 'metrics'

checkpoints_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

print('=' * 75)
print('STARTING PHASE 3 CNN + GRU HYBRID TRAINING ON CUDA GPU')
print('=' * 75)
print(f'Compute Device      : {compute_device}')
print(f'Batch Size          : {BATCH_SIZE}')
print(f'Max Epochs          : {MAX_EPOCHS}')
print(f'Early Stop Patience : {PATIENCE}')
print(f'Learning Rate       : {LEARNING_RATE}')
print('=' * 75)

train_results = {}
for seed in SEEDS:
    print(f'\n>>> Training CNN + GRU Hybrid (Seed {seed}) <<<')
    t0 = time.time()
    summary = train_cnn_gru_model(
        processed_dir=processed_dir,
        checkpoints_dir=checkpoints_dir,
        figures_dir=figures_dir,
        metrics_dir=metrics_dir,
        seed=seed,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        device=compute_device,
    )
    duration = time.time() - t0
    train_results[seed] = summary
    print(f'Seed {seed} finished in {duration:.1f}s | Best Val Macro-F1: {summary["best_val_macro_f1"]:.4f} (Epoch {summary["best_epoch"]})')

    # Sync checkpoint and learning curve figure to Google Drive
    if GDRIVE_DIR:
        ckpt_name = f'cnn_gru_seed{seed}.pt'
        fig_name = f'fig_phase3_cnn_gru_curves_seed{seed}.png'
        meta_name = f'phase3_cnn_gru_train_seed{seed}.json'
        shutil.copy2(checkpoints_dir / ckpt_name, GDRIVE_DIR / 'checkpoints' / ckpt_name)
        shutil.copy2(figures_dir / fig_name, GDRIVE_DIR / 'results' / 'figures' / fig_name)
        shutil.copy2(metrics_dir / meta_name, GDRIVE_DIR / 'results' / 'metrics' / meta_name)
        print(f'Persisted Seed {seed} artifacts directly to Google Drive.')

print('\nPhase 3 training routine successfully completed.')

In [ ]:
# CELL 6: Held-Out Fold 10 Evaluation strictly once
from src.train_hybrid import evaluate_cnn_gru_model

primary_seed = SEEDS[0]
ckpt_path = checkpoints_dir / f'cnn_gru_seed{primary_seed}.pt'

print('=' * 75)
print(f'EVALUATING FINAL MODEL ON HELD-OUT TEST FOLD 10 (Seed {primary_seed}, N=837)')
print('=' * 75)

test_metrics = evaluate_cnn_gru_model(
    checkpoint_path=ckpt_path,
    processed_dir=processed_dir,
    metrics_dir=metrics_dir,
    seed=primary_seed,
    threshold=0.50,
    device=compute_device,
)
test_metrics['val_macro_f1'] = train_results[primary_seed]['best_val_macro_f1']

cm = test_metrics['confusion_matrix']
print(f'Test Macro-F1       : {test_metrics["macro_f1"]:.4f}')
print(f'Siren F1 Score      : {test_metrics["siren_f1"]:.4f}')
print(f'Siren Recall        : {test_metrics["siren_recall"] * 100:.2f}% ({cm["tp"]}/{cm["tp"] + cm["fn"]})')
print(f'Siren Precision     : {test_metrics["siren_precision"] * 100:.2f}% ({cm["tp"]}/{cm["tp"] + cm["fp"]})')
print(f'PR-AUC              : {test_metrics["pr_auc"]:.4f}')
print(f'ROC-AUC             : {test_metrics["roc_auc"]:.4f}')
print(f'Balanced Accuracy   : {test_metrics["balanced_accuracy"] * 100:.2f}%')
print(f'Inference Latency   : {test_metrics["inference_latency_ms"]:.3f} ms / sample (on {compute_device})')
print(f'Confusion Matrix    : TP={cm["tp"]}, FP={cm["fp"]}, TN={cm["tn"]}, FN={cm["fn"]}')

# Persist test metrics to Drive
if GDRIVE_DIR:
    test_json = metrics_dir / f'phase3_cnn_gru_test_seed{primary_seed}.json'
    shutil.copy2(test_json, GDRIVE_DIR / 'results' / 'metrics' / test_json.name)
    print(f'Saved test metrics to Google Drive: {test_json.name}')

In [ ]:
# CELL 7: Multi-Model Architectural Benchmark & Comparison Table
from src.train_hybrid import build_phase3_comparison_table

comparison_df = build_phase3_comparison_table(
    metrics_dir=metrics_dir,
    figures_dir=figures_dir,
    cnn_gru_test_metrics=test_metrics,
)

print('\n' + '=' * 105)
print('PHASE 3 ARCHITECTURAL BENCHMARK: DEEP MLP vs. 2D CNN vs. CNN + GRU HYBRID (FOLD 10)')
print('=' * 105)
print(comparison_df.to_markdown(index=False))
print('=' * 105)

# Save comparison table CSV and figures to Google Drive
csv_path = metrics_dir / 'phase3_model_comparison.csv'
comparison_df.to_csv(csv_path, index=False)
fig_path = figures_dir / 'fig_phase3_architecture_comparison.png'

if GDRIVE_DIR:
    shutil.copy2(csv_path, GDRIVE_DIR / 'results' / 'metrics' / csv_path.name)
    if fig_path.is_file():
        shutil.copy2(fig_path, GDRIVE_DIR / 'results' / 'figures' / fig_path.name)
    print('Saved benchmark CSV and comparison figure to Google Drive.')

In [ ]:
# CELL 8: Downloadable Artifact Summary & Local Import Instructions
import zipfile

out_zip_path = WORKSPACE / 'phase3_trained_artifacts.zip'
artifact_files = [
    checkpoints_dir / f'cnn_gru_seed{primary_seed}.pt',
    figures_dir / f'fig_phase3_cnn_gru_curves_seed{primary_seed}.png',
    figures_dir / 'fig_phase3_architecture_comparison.png',
    metrics_dir / f'phase3_cnn_gru_train_seed{primary_seed}.json',
    metrics_dir / f'phase3_cnn_gru_test_seed{primary_seed}.json',
    metrics_dir / 'phase3_model_comparison.csv',
]

with zipfile.ZipFile(out_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for f in artifact_files:
        if f.is_file():
            arcname = f.relative_to(WORKSPACE).as_posix()
            z.write(f, arcname=arcname)
            print(f'Bundled: {arcname}')

if GDRIVE_DIR:
    drive_artifact_zip = GDRIVE_DIR / 'phase3_trained_artifacts.zip'
    shutil.copy2(out_zip_path, drive_artifact_zip)
    print(f'\nTrained artifacts packaged to Google Drive: {drive_artifact_zip}')

print('\n' + '=' * 80)
print('HOW TO IMPORT TRAINED CHECKPOINT TO LOCAL WINDOWS PC')
print('=' * 80)
print('1. Download `phase3_trained_artifacts.zip` from your Google Drive folder:')
print('   MyDrive/ANN-Siren-Detection/phase3_trained_artifacts.zip')
print('2. On your Windows machine at D:\\ANN-Siren-Detection, extract the zip into the project root:')
print('   Expand-Archive -Path phase3_trained_artifacts.zip -DestinationPath D:\\ANN-Siren-Detection -Force')
print('3. Verify the newly imported model in the Streamlit application:')
print('   streamlit run app.py')
print('   Or test inference directly:')
print('   python -c "from src.inference import SirenInferenceEngine; engine = SirenInferenceEngine(); print(engine.available_models)"')
print('=' * 80)